In [1]:
import qutip as qt
from qutip import tensor, basis
import numpy as np
import matplotlib.pyplot as plt
from quantum_logical.channel import AmplitudeDamping, PhaseDamping
from quantum_logical.trotter import TrotterGroup
from tqdm import tqdm
from quantum_logical.operators import selective_destroy
from scipy.optimize import curve_fit

In [2]:
# generating parameters and creating initial state
T1 = 50
T2 = 25
N = 5
dim = 3
trotter_dt = .01
amp_damp_channel = AmplitudeDamping(T1, num_qubits=N, hilbert_space_dim=dim)
phase_damp_channel = PhaseDamping(T1, T2, num_qubits=N, hilbert_space_dim=dim)
trotter = TrotterGroup(
    continuous_operators=[amp_damp_channel, phase_damp_channel],
    trotter_dt=trotter_dt,
)


The qutrit subspace of | g $\rangle$ and | f $\rangle$ $\newline$
The qutrit error state is the | e $\rangle$

In [3]:
# creating the initial state of the system 
psi0 = tensor(basis(dim, 2), basis(dim, 0), basis(dim, 0), basis(dim, 0), basis(dim, 0))
rho0 = psi0 * psi0.dag()

In [13]:
# creating the equivalent two qubit gates
# encoding set 
cnot1 = qt.cnot(N=5, target=1, control=0)
cnot2 = qt.cnot(N=5, target=2, control=0)

hadamard = (1 / np.sqrt(2)) * qt.Qobj([[1, 0, 1], [0, np.sqrt(2), 0], [1, 0, -1]])  # this has been put in the qutrit basis 
hadamard_layer = tensor(tensor([hadamard] * 3) , tensor([qt.qeye(dim)] * 2))

# stabilizer set
cnot3 = qt.cnot(N=5, target=3, control=0)
cnot4 = qt.cnot(N=5, target=3, control=1)

cnot5 = qt.cnot(N=5, target=4, control=1)
cnot6 = qt.cnot(N=5, target=4, control=2)

# correction set 
# creating the z gates (this is not hard)
hadamard_layer1 = hadamard_layer.full()
hadamard_layer1 = qt.Qobj(hadamard_layer1)

/var/folders/nc/8sqqbv456mdfx096ks86vy8m0000gn/T/ipykernel_83170/1035679642.py:3: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot1 = qt.cnot(N=5, target=1, control=0)
/var/folders/nc/8sqqbv456mdfx096ks86vy8m0000gn/T/ipykernel_83170/1035679642.py:4: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot2 = qt.cnot(N=5, target=2, control=0)
/var/folders/nc/8sqqbv456mdfx096ks86vy8m0000gn/T/ipykernel_83170/1035679642.py:10: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule inste

Expanding gates into a larger dimensionality is easy $\newline$
To create the change matrix build an identity in the subspace of the basis you want to go from to the one you want to go to $\newline$
The important thing here is that the matrix should be of dimensionality m x n where m is the dimensionality you are moving from to the one you are going to $\newline$
Once this is done then you multiply these out and then add in the a matrix of the added dimension vector $\newline$
INSERT AN IMAGE FOR COMPREHENSION

In [5]:
# making a list 
# this makes the matricies for the isometry this is not hard 
# now the gates can be made from it and this is not hard 

def create_higher_dim(N, new_dim, old_dim, choice, lower_dim_gate): # can only add one level at a time
    nums_block = []
    if choice == "add":
        matrix_make = np.zeros((old_dim ** N, new_dim ** N))
        for i in range(N): 
            nums_block.append(i * new_dim + old_dim)
 
        list1 = []
        for i in range(old_dim ** N):
                list1.append(i)


        list2 = [] 
        for j in range(old_dim ** N + old_dim):
            if j not in nums_block:
                list2.append(j)

        for i in range(len(list1)):
            matrix_make[list1[i], list2[i]] = 1

        a = qt.Qobj(matrix_make)
        a = a.dag() * lower_dim_gate * a

        new_mat = np.zeros((new_dim ** N, new_dim ** N))
        for i in range(dim ** N):
            if i not in list2:
                new_mat[i,i] = 1
        
        b = qt.Qobj(new_mat)

        new_gate = a + b


    return new_gate # returns the new higher dim gate
# this may be doable if you can preserve the dimensionality 
# so far this wont work because you did not preserve the dimensionality 

In [30]:
# redefine the function above into something that makes more sense
# for this to work you need to decompose the incoming gate 
old_gate_array = []
N = 2
x_gate = tensor([qt.Qobj([[0,1], [1,0]])] * N)
for i in range(N):
    old_gate_array.append(qt.ptrace(x_gate, [i]))
create_mat = np.zeros((2, 3))
for i in range(2):
    create_mat[i,i] = 1
    # the small matrix is not created 
    a = qt.Qobj(create_mat)

new_gate_array = []
for i in range(len(old_gate_array)):
    new_gate_array.append(a.dag() * old_gate_array[i] * a)

tensor(new_gate_array)

x_gate=tensor(a,a).dag() * x_gate * tensor(a , a)
x_gate

Quantum object: dims = [[3, 3], [3, 3]], shape = (9, 9), type = oper, isherm = True
Qobj data =
[[0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

In [6]:
create_higher_dim(1, new_dim=3, old_dim=2, choice="add", lower_dim_gate=qt.qeye(2)) # send these as full Qobj without dim break down

Quantum object: dims = [[3], [3]], shape = (3, 3), type = oper, isherm = True
Qobj data =
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

In [ ]:
new_dim = 3
for i in range(new_dim):
    for j in range(new_dim):
        if i != new_dim and j == new_dim:
            a = tensor(basis(new_dim, i), basis(new_dim, j)) * tensor(basis(new_dim, i), basis(new_dim, j)).dag()
        if i == new_dim:
            a = tensor(basis(new_dim, i), basis(new_dim, j)) * tensor(basis(new_dim, i), basis(new_dim, j)).dag()
print(a)

In [ ]:
# build the set of gates that extend the way you need them to 


In [ ]:
# start building the circuits now that the gates have been built 